In [ ]:
# ======================================================================
# CLEAN DUBLIN BER: one ML-ready table, one row per Dublin home
# ======================================================================
# Input:  the 19 Dublin processed BER tables
#         (just_dublin_dataset\just_dublin_processed_dataset\ber)
# Output: just_dublin_dataset\Just_dublin_clean_dataset\ber\ber_dublin_clean.csv
#
# Same cleaning as the national BER notebook, but for Dublin only and
# keeping the location columns:
#   nation, province, county, local_authority (empty for BER), dublin_district
#
# What this table is for (Student 1):
#   - K-Prototypes clustering + FAMD   -> building features
#   - XGBoost regression               -> target: BerRating (kWh/m2/yr)
#   - Random Forest classification     -> target: is_B3_or_better
#
# Rules: fix the data, do NOT fill missing values / scale / encode here,
# and leave out leakage columns (outputs of the BER calculation).
#
# Needs the Dublin processed notebooks (00 and 01) to be run first.
# ======================================================================

import pandas as pd
import numpy as np
import os

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

base = r'C:\Users\LENOVO\OneDrive\OneDrive - National College of Ireland\Machine_learning_Project'

ber_folder   = base + r'\just_dublin_dataset\just_dublin_processed_dataset\ber'
dublin_clean = base + r'\just_dublin_dataset\Just_dublin_clean_dataset'

In [ ]:
# Loading only the tables that hold useful inputs or the targets.
# Left out on purpose:
#   group_heating, group_boilers, other_energy, solar_pv, wind_turbines, floors
#   -> mostly empty or already summed up in other columns
#   most of results -> CO2Rating, EPC, CPC, RER, primary energy columns,
#   HeatLossIndicator are calculated from the same energy calculation as
#   BerRating, so using them as inputs would be leakage

def load(name):
    return pd.read_csv(ber_folder + '\\dublin_' + name + '.csv', low_memory=False)

assessment    = load('assessment')
results       = load('results')[['ber_id', 'BerRating', 'EnergyRating']]
geometry      = load('geometry')
fabric        = load('fabric')
ventilation   = load('ventilation')
space_heating = load('space_heating')
water_heating = load('water_heating')
solar_water   = load('solar_water')[['ber_id', 'SolarHotWaterHeating']]
renewables    = load('nzeb_renewables')[['ber_id', 'SolarPVEnergyOutput']]
walls         = load('walls')

ber_county    = load('county')
dwelling_type = load('dwelling_type')
fuel          = load('fuel')

print('assessment:', assessment.shape)
print('walls:     ', walls.shape)

In [ ]:
# Picking the columns that describe the building itself.
# These are the things an assessor records about the house: size, insulation,
# ventilation, heating system. None of them are results of the rating.

# The location columns made in the Dublin processed notebook are kept too.

assessment = assessment[['ber_id', 'nation', 'province', 'county', 'county_id', 'local_authority',
                         'dublin_district', 'district_id', 'ber_location',
                         'dwelling_type_id', 'Year_of_Construction',
                         'TypeofRating', 'PurposeOfRating', 'DateOfAssessment']]

geometry = geometry[['ber_id', 'GroundFloorArea(sq m)', 'NoStoreys', 'WallArea', 'RoofArea',
                     'FloorArea', 'WindowArea', 'DoorArea', 'LivingAreaPercent',
                     'RoomInRoofArea', 'PredominantRoofType']]

fabric = fabric[['ber_id', 'UValueWall', 'UValueRoof', 'UValueFloor', 'UValueWindow', 'UvalueDoor',
                 'ThermalBridgingFactor', 'ThermalMassCategory', 'StructureType',
                 'SuspendedWoodenFloor', 'PercentageDraughtStripped', 'NoOfSidesSheltered',
                 'PermeabilityTest']]

ventilation = ventilation[['ber_id', 'NoOfChimneys', 'NoOfOpenFlues', 'NoOfFansAndVents',
                           'NoOfFluelessGasFires', 'DraftLobby', 'VentilationMethod',
                           'HeatExchangerEff']]

space_heating = space_heating[['ber_id', 'main_fuel_id', 'HSMainSystemEfficiency',
                               'HSSupplHeatFraction', 'HeatSystemControlCat',
                               'HeatSystemResponseCat', 'UndergroundHeating',
                               'WarmAirHeatingSystem', 'CombiBoiler']]
space_heating = space_heating.rename(columns={'main_fuel_id': 'space_fuel_id'})

water_heating = water_heating[['ber_id', 'main_fuel_id', 'WHMainSystemEff', 'WaterStorageVolume',
                               'InsulationType', 'CylinderStat']]
water_heating = water_heating.rename(columns={'main_fuel_id': 'water_fuel_id'})

In [ ]:
# A house can have up to 3 wall types. For one row per BER, keeping the
# main wall type = the one with the biggest area.

walls = walls.sort_values(['ber_id', 'wall_area'], ascending=[True, False])
main_wall = walls.drop_duplicates(subset='ber_id', keep='first')
main_wall = main_wall[['ber_id', 'wall_type']].rename(columns={'wall_type': 'MainWallType'})

print('BERs with a main wall type:', len(main_wall))
main_wall.head()

In [ ]:
# Joining everything back into one table using ber_id (like a SQL join).
# how='left' keeps every BER from the assessment table.

ber = assessment.merge(results, on='ber_id', how='left')
ber = ber.merge(geometry, on='ber_id', how='left')
ber = ber.merge(fabric, on='ber_id', how='left')
ber = ber.merge(ventilation, on='ber_id', how='left')
ber = ber.merge(space_heating, on='ber_id', how='left')
ber = ber.merge(water_heating, on='ber_id', how='left')
ber = ber.merge(solar_water, on='ber_id', how='left')
ber = ber.merge(renewables, on='ber_id', how='left')
ber = ber.merge(main_wall, on='ber_id', how='left')

print(ber.shape)   # should still be 569,058 rows

In [ ]:
# Putting the names back in place of the ids (dwelling type and fuels),
# so the table is readable and the models can encode them later.

type_map = dict(zip(dwelling_type['dwelling_type_id'], dwelling_type['DwellingTypeDescr']))
fuel_map = dict(zip(fuel['fuel_id'], fuel['FuelName']))

ber['DwellingType']         = ber['dwelling_type_id'].map(type_map)
ber['MainSpaceHeatingFuel'] = ber['space_fuel_id'].map(fuel_map)
ber['MainWaterHeatingFuel'] = ber['water_fuel_id'].map(fuel_map)

ber = ber.drop(columns=['dwelling_type_id', 'space_fuel_id', 'water_fuel_id'])

print(ber['DwellingType'].value_counts(dropna=False))
print(ber['MainSpaceHeatingFuel'].value_counts(dropna=False))

In [ ]:
# Location check: every Dublin home should have a district
# (a real one, or 'Dublin (no district)' for homes recorded as 'Co. Dublin').

print('Rows without a district:', ber['dublin_district'].isnull().sum())   # should be 0
print(ber['dublin_district'].value_counts())

In [ ]:
# Text columns: SEAI pads a lot of values with spaces at the end
# ('Existing       ', 'NO  ', 'Masonry   '). Removing them.
# Also some 'missing' values are hidden as text: '' or 'Please select'.
# Turning those into real missing values (NaN).

text_cols = []
for col in ber.columns:
    if ber[col].dtype == 'object' or str(ber[col].dtype) == 'str':
        text_cols.append(col)

for col in text_cols:
    ber[col] = ber[col].str.strip()
    ber[col] = ber[col].replace(['', 'Please select'], np.nan)

print('Text columns cleaned:', text_cols)

In [ ]:
# YES / NO columns -> 1 / 0, so models can use them directly.
# Only changing columns where every value is YES, NO, Yes or No.

yes_no_cols = []
for col in text_cols:
    values = set(ber[col].dropna().unique())
    if len(values) > 0 and values.issubset({'YES', 'NO', 'Yes', 'No'}):
        ber[col] = ber[col].map({'YES': 1, 'Yes': 1, 'NO': 0, 'No': 0})
        yes_no_cols.append(col)

print('Changed to 1/0:', yes_no_cols)

In [ ]:
# Dates come back as text from CSV, so converting again.
# Adding year and quarter (e.g. 2021Q1) in the same format as PPR, rent and
# planning, for the county + quarter join later.

ber['DateOfAssessment'] = pd.to_datetime(ber['DateOfAssessment'])
ber['year'] = ber['DateOfAssessment'].dt.year
ber['quarter'] = ber['DateOfAssessment'].dt.to_period('Q').astype(str)

print(ber['year'].value_counts().sort_index())

In [ ]:
# Year of construction goes up to 2073, which is impossible.
# A few 2026 values are fine (provisional ratings for homes not built yet),
# so only years after 2026 become missing.

print('Years after 2026:', (ber['Year_of_Construction'] > 2026).sum())
print(ber[ber['Year_of_Construction'] > 2026][['ber_id', 'Year_of_Construction', 'TypeofRating']])

ber.loc[ber['Year_of_Construction'] > 2026, 'Year_of_Construction'] = np.nan

In [ ]:
# Looking at the energy rating labels after removing spaces.
# Expected the usual 15 bands (A1 ... G), but there are also A, A0, B, C, D, E.

print(ber['EnergyRating'].value_counts().sort_index())

In [ ]:
# The odd labels only show up from 2021 and grow a lot in 2024-2025,
# and their BerRating values sit in clear ranges (A0 lowest, then A, B, C...).
# So they look like a different (newer) rating scale, not typing mistakes.
# Keeping the rows, but marking which scale each label belongs to.

old_scale = ['A1', 'A2', 'A3', 'B1', 'B2', 'B3', 'C1', 'C2', 'C3',
             'D1', 'D2', 'E1', 'E2', 'F', 'G']

ber['rating_scale'] = 'other'
ber.loc[ber['EnergyRating'].isin(old_scale), 'rating_scale'] = 'old_15_band'

print(ber['rating_scale'].value_counts())
print()
print(ber.groupby(['EnergyRating'])['BerRating'].agg(['count', 'min', 'max']).sort_values('min'))

In [ ]:
# Classification target: is the home B3 or better?
# Homes on the 15-band scale: A1, A2, A3, B1, B2, B3 -> 1, everything else -> 0
# (using the letters here, because a few homes sit exactly on 150 and the
#  letter is the official answer for them)
# Homes on the other scale (A0, A, B, C, D, E) have no B3 letter, so for those
# the BerRating cut-off is used instead: B3 or better = up to 150 kWh/m2/yr.

b3_or_better = ['A1', 'A2', 'A3', 'B1', 'B2', 'B3']

ber['is_B3_or_better'] = 0
ber.loc[ber['EnergyRating'].isin(b3_or_better), 'is_B3_or_better'] = 1
ber.loc[(ber['rating_scale'] == 'other') & (ber['BerRating'] <= 150), 'is_B3_or_better'] = 1

# Checking the labels
print(pd.crosstab(ber['EnergyRating'], ber['is_B3_or_better']))
print()
print(ber['is_B3_or_better'].value_counts(normalize=True).round(3))

In [ ]:
# BerRating range check. Negative values are homes that make more energy than
# they use (e.g. solar panels), so they are real and kept.
# Very high values are kept too; outliers are dealt with at the modelling stage.

print(ber['BerRating'].describe().round(2))
print('Negative BerRating:', (ber['BerRating'] < 0).sum())

In [ ]:
# Tidying column names and order: keys first, then building features,
# then the targets at the end.

ber = ber.rename(columns={'GroundFloorArea(sq m)': 'TotalFloorArea'})

key_cols = ['ber_id', 'nation', 'province', 'county', 'county_id', 'local_authority',
            'dublin_district', 'district_id', 'ber_location', 'DateOfAssessment', 'year', 'quarter']
target_cols = ['BerRating', 'EnergyRating', 'rating_scale', 'is_B3_or_better']

feature_cols = []
for col in ber.columns:
    if col not in key_cols and col not in target_cols:
        feature_cols.append(col)

ber = ber[key_cols + feature_cols + target_cols]

print(len(feature_cols), 'feature columns:')
print(feature_cols)

In [ ]:
# Missing values left in the clean table (% empty per column).
# Not filling them here; the model Pipeline will handle them on training data.

missing = (ber.isnull().mean() * 100).round(1)
missing[missing > 0].sort_values(ascending=False)

In [ ]:
ber.info()
ber.head()

In [ ]:
# Saving the clean Dublin BER table.

save_folder = dublin_clean + r'\ber'
os.makedirs(save_folder, exist_ok=True)

ber.to_csv(save_folder + r'\ber_dublin_clean.csv', index=False)
print('Saved', ber.shape, 'to', save_folder + r'\ber_dublin_clean.csv')